# 📝 그래프 집계 과제 LV2(응용): 랭킹·HAVING·제약·2단 파이프라인·max·SET·레이블별 인덱스

> LV1 의 집계를 **조합**합니다. 상위 N 랭킹(보조키), 두 관계를 잇는 합산, 집계 후 필터(HAVING), collect 목록, UNIQUE 제약과 중복 판별, **2단 WITH 파이프라인**, **최댓값(max)과 파생 속성(SET)**, 그리고 마지막으로 **인덱스가 레이블마다 따로 걸린다**는 것을 실행계획으로 확인합니다.

## 풀이 방법
1. 맨 위 **준비 셀 3개**(연결 → 초기화 → 시드 적재)를 먼저 실행하세요. 반드시 **실습 전용 DB**로.
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).
3. 결과(dict 리스트)를 지정한 변수(대개 `rows`)에 담으세요.

- 도메인: **음악 차트**: 청취자(Listener)가 곡(Song)을 재생(PLAYED{cnt})하고, 아티스트(Artist)가 곡을 공연(PERFORMS)합니다.

화이팅!

아래 준비 셀이 만들 그래프의 전체 모습입니다. 아티스트 3명·곡 6개·청취자 4명이 두 종류의 관계로 이어져 있습니다. 곡마다 붙은 **`tag`** 는 외부 시스템에서 받은 원본 문자열 그대로입니다(15번에서 다듬어 씁니다).

<img src="images/그래프_한눈에_차트.png" width="820">

아래 준비 셀 3개를 위에서부터 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. .env 로 연결하고 run_cypher 헬퍼를 만듭니다.
# 반드시 "실습 전용" DB 에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # UNIQUE 제약 위반 에러

# 1) 접속 정보: .env 를 환경변수로 올린다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러가 아니라 기본값으로 넘어간다. 마지막 줄의 주소를 눈으로 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북이 끝날 때까지 재사용할 통로 하나
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 키는 RETURN 의 별칭이다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요!
# 노드·관계에 더해 이 노트북이 만든 인덱스·제약까지 지웁니다(DB 기본 LOOKUP 인덱스는 그대로).
# 1) 제약 먼저. 제약이 남아 있으면 그 제약이 만든 인덱스를 따로 못 지운다
for _row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _row["name"] + " IF EXISTS")
# 2) 남은 인덱스. LOOKUP 은 DB 기본이라 뺀다
for _row in run_cypher("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP' RETURN name"):
    run_cypher("DROP INDEX " + _row["name"] + " IF EXISTS")
# 3) 노드·관계. 관계가 9만 개라 한 번에 담지 않고 2만 개씩 끊어 지운다
while True:
    # DETACH DELETE 는 매달린 관계까지 함께 지운다
    _left = run_cypher("MATCH (n) WITH n LIMIT 20000 DETACH DELETE n RETURN count(n) AS n")[0]["n"]
    if _left == 0:
        break
print("초기화 완료: 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 음악 차트 시드 적재: 실행만 하세요.
# Artist -PERFORMS-> Song{tag} <-PLAYED{cnt}- Listener
# tag 는 외부 시스템에서 받은 원본 문자열 그대로다("장르|발매연도" + 앞뒤 공백, 15번에서 다룬다).
songs = [
    ["은하수", "루나", " 발라드|2021 "], ["밤하늘", "루나", "발라드|2019 "],
    ["파도", "제이드", " 댄스|2022"], ["모래성", "제이드", " 발라드|2020 "],
    ["등대", "제이드", "댄스|2023 "],
    ["질주", "카이", " 록|2022 "],
]
listeners = ["하늘", "바다", "별", "산"]
played = [
    ["하늘", "은하수", 50], ["하늘", "파도", 30],
    ["바다", "은하수", 40], ["바다", "등대", 60],
    ["별", "밤하늘", 20], ["별", "파도", 45], ["별", "질주", 35],
    ["산", "모래성", 25], ["산", "등대", 55], ["산", "은하수", 15],
]
for title, artist, tag in songs:
    run_cypher("MERGE (a:Artist {name: $artist}) "
               "MERGE (s:Song {title: $title}) SET s.tag = $tag "
               "MERGE (a)-[:PERFORMS]->(s)", artist=artist, title=title, tag=tag)
for name in listeners:
    run_cypher("MERGE (:Listener {name: $name})", name=name)
for listener, title, cnt in played:
    run_cypher("MATCH (l:Listener {name: $listener}), (s:Song {title: $title}) "
               "MERGE (l)-[x:PLAYED]->(s) SET x.cnt = $cnt", listener=listener, title=title, cnt=cnt)
print("적재 완료: 곡:", run_cypher("MATCH (s:Song) RETURN count(s) AS n")[0]["n"],
      "· 재생:", run_cypher("MATCH ()-[x:PLAYED]->() RETURN count(x) AS n")[0]["n"])


## 그래프 살펴보기

문제로 들어가기 전에 **무엇이 들어 있는지** 한 번 훑습니다. 어떤 레이블이 몇 개인지, 어떤 관계가 몇 개이고 거기에 어떤 속성이 붙어 있는지를 알아야 "무엇을 무엇으로 묶을지"를 정할 수 있습니다.

In [ ]:
# [제공 코드] 그래프에 무엇이 들어 있는지 훑어봅니다(실행만 하세요).
# 1) 레이블마다 몇 개인지. 노드마다 레이블이 하나뿐이라 labels(n)[0] 로 충분하다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS 레이블, count(*) AS 개수 ORDER BY 레이블"):
    print(row)

# 2) 관계 타입마다 몇 개인지. type(x) 가 관계 종류 이름이다
for row in run_cypher("MATCH ()-[x]->() RETURN type(x) AS 관계, count(x) AS 개수 ORDER BY 관계"):
    print(row)

# 3) 관계에 붙은 속성 이름. keys(x) 가 그 목록이다
for row in run_cypher("MATCH ()-[x]->() "
                      "RETURN type(x) AS 관계, collect(DISTINCT keys(x)) AS 속성키 ORDER BY 관계"):
    print(row)


## 1. 재생수 상위 3곡 (랭킹 + LIMIT)
**배경**: 곡별 총 재생수(`cnt` 합)를 구해 상위 3곡을 뽑습니다.

**요구사항**:
- `(:Listener)-[p:PLAYED]->(s:Song)` 에서 곡별 `sum(p.cnt)` 를 구해, **총재생 내림차순**(같으면 곡 제목 오름차순)으로 정렬하고 `LIMIT 3` 으로 상위 3곡만 **`rows`** 에 담으세요. 별칭은 **`곡`**·**`총재생`**.

**예시**: `rows[0]` 은 곡 **등대**, 총재생 **115** 이고, 상위 3곡은 **['등대', '은하수', '파도']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 곡 제목을 그룹핑 키로 두어 재생 관계의 cnt 를 sum 으로 합하고, 정렬 후 LIMIT 로 상위 3만 남긴다.

세부구현:
1. MATCH 로 청취자→곡(PLAYED) 을 잡고 재생 관계에 변수를 붙인다.
2. RETURN 에 곡 제목과 sum 집계를 별칭(곡·총재생)으로 둔다.
3. 총재생 내림차순(같으면 곡 제목순)으로 ORDER BY 하고 LIMIT 로 상위 3만 남긴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert [r['곡'] for r in rows] == ['등대', '은하수', '파도'], '상위 3곡이 다릅니다. 총재생 내림차순(같으면 제목순)으로 정렬하고 LIMIT 3 했는지 확인하세요'
assert rows[0]['총재생'] == 115, \
    '1위 곡의 총재생이 다릅니다. 곡마다 재생 관계의 cnt 를 sum 으로 합했는지 확인하세요'
print('✅ 통과!')

## 2. 아티스트별 총 재생수 (두 관계 합산)
**배경**: 아티스트가 공연한 곡들의 재생수를 모두 합쳐 아티스트별 인기를 봅니다. `PERFORMS` 와 `PLAYED` 두 관계를 한 패턴으로 잇습니다.

**요구사항**:
- `(a:Artist)-[:PERFORMS]->(s:Song)<-[p:PLAYED]-(:Listener)` 에서 아티스트별 `sum(p.cnt)` 를 구해 **`rows`** 에 담으세요. 별칭은 **`아티스트`**·**`총재생`**.
- **총재생 내림차순**(같으면 아티스트 이름 오름차순)으로 정렬하세요.

**예시**: `rows[0]` 은 아티스트 **제이드**, 총재생 **215** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- PERFORMS(아티스트→곡)와 PLAYED(청취자→곡)를 한 패턴으로 잇고, 아티스트 이름을 그룹핑 키로 둔다.

세부구현:
1. MATCH 로 아티스트→곡(PERFORMS)과 그 곡의 재생(PLAYED)을 한 패턴으로 잇고, 재생 관계에 변수를 붙인다.
2. RETURN 에 아티스트 이름과 sum 집계를 별칭(아티스트·총재생)으로 둔다.
3. 총재생 내림차순(같으면 아티스트 이름순)으로 ORDER BY 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 3, '아티스트 3명이 모두 나와야 합니다'
assert rows[0]['아티스트'] == '제이드' and rows[0]['총재생'] == 215, '1위가 다릅니다. 아티스트-곡-재생 두 관계를 이어 sum 했는지 확인하세요'
assert rows == sorted(rows, key=lambda r: (-r['총재생'], r['아티스트'])), '정렬을 확인하세요: 총재생 내림차순, 같으면 아티스트 이름 오름차순'
print('✅ 통과!')

## 3. 총 재생수 100 이상인 곡 (집계 후 필터)
**배경**: "총 재생수 100 이상"은 **집계 결과에 대한 조건**이라 `WITH` 로 먼저 합친 뒤 걸러야 합니다.

**요구사항**:
- 곡별 `sum(p.cnt)` 를 집계한 뒤 **100 이상**인 곡만 남겨 **`rows`** 에 담으세요. 별칭은 **`곡`**·**`총재생`**.
- **총재생 내림차순**(같으면 곡 제목 오름차순)으로 정렬하세요.

**예시**: 조건을 만족하는 곡은 **2개**이고 `rows[0]` 은 **등대** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 집계 조건이므로 WITH 로 곡별 sum 을 먼저 낸 뒤, 이어지는 WHERE 로 100 이상만 남긴다.

세부구현:
1. MATCH 로 청취자→곡(PLAYED) 을 잡고 재생 관계에 변수를 붙인다.
2. WITH 로 곡과 sum 집계(별칭 총재생)를 넘기고, 이어지는 WHERE 로 총재생 100 이상만 남긴다.
3. RETURN 에 곡 제목·총재생을 두고 총재생 내림차순(같으면 곡 제목순)으로 ORDER BY 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 2, '2행이어야 합니다. 조건을 WITH 뒤 WHERE 에 걸었는지 확인하세요'
assert rows[0]['곡'] == '등대', '1위 곡이 다릅니다. 곡별 sum 으로 집계했는지 확인하세요'
assert all(r['총재생'] >= 100 for r in rows), '100 미만인 곡이 남아 있습니다'
print('✅ 통과!')

## 4. 아티스트의 곡 목록 (collect)
**배경**: 아티스트 **`제이드`** 가 공연한 곡 제목을 리스트로 모읍니다.

**요구사항**:
- 아티스트 `제이드` 가 `PERFORMS` 하는 곡 제목을 `collect` 로 모아 **`rows`** 에 담으세요. 별칭은 **`곡목록`**.

**예시**: `rows[0]['곡목록']` 을 정렬하면 **['등대', '모래성', '파도']** 입니다(리스트 순서는 상관없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 아티스트 이름을 제이드로 못박아 필터한 뒤(패턴의 노드 속성), 곡 제목을 collect 로 모은다.

세부구현:
1. MATCH 로 아티스트(이름=제이드)→곡(PERFORMS) 패턴을 잡는다.
2. RETURN 에 곡 제목을 collect 한 집계를 별칭(곡목록)으로 둔다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sorted(rows[0]['곡목록']) == ['등대', '모래성', '파도'], '곡 목록이 다릅니다. 제이드로 좁혔는지, collect 로 제목을 모았는지 확인하세요'
print('✅ 통과!')

## 5. 곡 제목 UNIQUE 제약과 중복 삽입 판별
**배경**: 같은 곡이 두 번 등록되지 않도록 `Song.title` 에 UNIQUE 제약을 걸고, 실제로 중복이 막히는지 확인합니다.

**요구사항**:
1. `Song.title` 에 UNIQUE 제약 **`song_title_unique`** 를 만드세요(`IF NOT EXISTS`) 후 `db.awaitIndexes()`.
2. 이미 있는 곡 제목 `은하수` 를 **또** `CREATE (:Song {title: '은하수'})` 해 보고, **제약 위반 에러**(`ConstraintError`)가 나면 변수 **`blocked`** 를 `True`, 안 나면 `False` 로 두세요(`try/except`). 잡은 예외의 **이름**(`type(오류).__name__`)도 변수 **`err_name`** 에 담으세요. `ConstraintError` 는 준비 셀에서 미리 import 해 두었습니다.

채점은 **제약이 실제로 등록됐는지도** 확인합니다. 제약을 만들지 않고 쿼리에 오타만 내도 에러는 나기 때문입니다(그건 제약이 막은 게 아닙니다).

> **순서를 지키세요.** 2번을 먼저 실행하면 `은하수` 가 두 개가 되고, 그 뒤로는 `CREATE CONSTRAINT` 가 계속 실패합니다(중복이 있으면 유일성 제약을 만들 수 없습니다). 그렇게 됐다면 `MATCH (s:Song {title: '은하수'}) WITH s LIMIT 1 DETACH DELETE s` 로 하나를 지운 뒤 1번부터 다시 하세요.

**예시**: 제약이 제대로 걸렸으면 중복 삽입이 막혀 `blocked` 는 **`True`** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안처럼 제약을 만든 뒤, 중복 CREATE 를 try 로 감싸 에러가 나면 blocked 를 True 로 둔다.

세부구현:
1. CREATE CONSTRAINT 로 Song 의 title 이 UNIQUE 하도록 제약(이름 song_title_unique, IF NOT EXISTS)을 만든다.
2. CALL db.awaitIndexes() 로 기다린다.
3. blocked 를 False, err_name 을 빈 문자열로 두고, try 안에서 이미 있는 제목('은하수')을 CREATE 해 본다.
4. 제약 위반 예외(ConstraintError)를 잡는 except 에서만 blocked 를 True 로 바꾸고, 잡은 오류의 __name__ 을 err_name 에 담는다 (아무 예외나 잡으면 오타로 난 문법 에러까지 성공으로 세게 된다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 제약이 실제로 등록됐는지부터 확인한다(에러가 났다는 사실만으로는 증거가 못 된다)
names = [r['name'] for r in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name")]
assert 'song_title_unique' in names, '제약 이름을 song_title_unique 로 만들었는지, SHOW CONSTRAINTS 결과를 담았는지 확인하세요'
# 채점이 직접 중복 삽입을 시도해 제약이 '정말' 막는지 확인한다(학생이 True 만 적었을 수도 있다)
try:
    run_cypher("CREATE (:Song {title: '은하수'})")
    grader_blocked = False
except ConstraintError:
    grader_blocked = True
assert grader_blocked, '제약이 중복 삽입을 막지 못했습니다. song_title_unique 를 만들고 db.awaitIndexes() 로 기다렸는지 확인하세요'
assert blocked is grader_blocked, '중복 삽입이 ConstraintError 로 막혔는지 확인하고 blocked 에 True 를 담으세요'
assert err_name == 'ConstraintError', '잡은 예외의 이름을 err_name 에 담으세요(except 절에서 type(오류).__name__). 제약 위반이면 ConstraintError 입니다'
# 대조: 제약은 '중복' 만 막아야지 새 곡 등록까지 막으면 안 된다
run_cypher("CREATE (:Song {title: '새벽'})")
assert run_cypher("MATCH (s:Song) RETURN count(s) AS n")[0]['n'] == 7, '제약이 새 곡 등록까지 막고 있습니다. title 에만 UNIQUE 를 걸었는지 확인하세요'
run_cypher("MATCH (s:Song {title: '새벽'}) DETACH DELETE s")   # 대조용으로 넣은 곡은 지운다
assert run_cypher("MATCH (s:Song {title: '은하수'}) RETURN count(s) AS n")[0]['n'] == 1, \
    '은하수 가 두 개 있습니다. 제약을 만들기 전에 중복을 넣었다면 하나를 지우고 다시 만드세요'
assert run_cypher("MATCH (s:Song) RETURN count(s) AS n")[0]['n'] == 6, \
    '곡이 6개가 아닙니다. 대조용으로 넣은 곡이 남아 있는지 확인하세요'
print('✅ 통과!')

## 6. 2단 파이프라인: 곡 합산 → 아티스트 합산
**배경**: 먼저 **곡별 재생수**를 낸 뒤, 그 결과를 아티스트로 다시 묶어 **아티스트별 재생수와 곡 수**를 함께 구합니다. `WITH` 를 두 번 거치는 파이프라인입니다.

**요구사항**:
- 1단계: 청취자→곡(`PLAYED`)을 잡아 **곡별로** 재생수를 `sum` 합치고, 그 곡별 합에 별칭 **`곡재생`** 을 붙여 `WITH` 로 넘깁니다.
- 2단계: 그 곡을 아티스트에 잇고(`PERFORMS`), **아티스트별로** `곡재생` 을 다시 `sum` 하고(별칭 **`아티스트재생`**) 곡 수도 `count` 로 셉니다(별칭 **`곡수`**).
- 결과를 **`rows`** 에 담고 **아티스트재생 내림차순**(같으면 아티스트 이름 오름차순)으로 정렬하세요. 아티스트 이름 별칭은 **`아티스트`**.

**예시**: `rows[0]` 은 아티스트 **제이드**, 아티스트재생 **215**, 곡수 **3** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 곡별 합(1단 WITH)을 낸 뒤, 그 곡을 아티스트에 이어 다시 합(2단 WITH)한다. 집계의 결과를 다음 집계의 입력으로 넘기는 게 핵심.

세부구현:
1. 청취자→곡(PLAYED)을 MATCH 하고, 첫 WITH 로 곡과 곡별 sum(재생수)을 별칭 곡재생 으로 넘긴다.
2. 그 곡을 아티스트에 잇는 MATCH(PERFORMS)를 이어 쓰고, 둘째 WITH 로 아티스트와 sum(곡재생)·count(곡)을 별칭 아티스트재생·곡수 로 넘긴다.
3. RETURN 에 아티스트 이름·아티스트재생·곡수를 두고 아티스트재생 내림차순(같으면 이름순)으로 ORDER BY 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows[0]['아티스트'] == '제이드', '1위 아티스트가 다릅니다'
assert rows[0]['아티스트재생'] == 215 and rows[0]['곡수'] == 3, '곡수가 재생 행 수로 세어졌을 수 있습니다. 1단 WITH 로 곡별 합을 먼저 냈는지 확인하세요'
assert len(rows) == 3, '아티스트 3명이 모두 나와야 합니다'
assert rows == sorted(rows, key=lambda r: (-r['아티스트재생'], r['아티스트'])), '정렬을 확인하세요: 아티스트재생 내림차순, 같으면 아티스트 이름 오름차순'
assert sum(r['곡수'] for r in rows) == 6, '곡수의 합이 전체 곡 수와 다릅니다. 2단 집계에서 곡을 중복해 세지 않았는지 확인하세요'
print('✅ 통과!')

## 7. 평균 재생수 35 이상인 아티스트 (평균 필터)
**배경**: 이번엔 합이 아니라 **평균**으로 거릅니다. 아티스트의 곡들이 받은 재생수(`cnt`)의 평균이 35 이상인 아티스트를 찾습니다.

**요구사항**:
- `(a:Artist)-[:PERFORMS]->(:Song)<-[p:PLAYED]-(:Listener)` 를 아티스트별 `avg(p.cnt)` 로 집계한 뒤, **평균 35 이상**만 남겨 **`rows`** 에 담으세요. 별칭은 **`아티스트`**·**`평균재생`**.
- **평균재생 내림차순**(같으면 아티스트 이름 오름차순)으로 정렬하세요.

**예시**: 조건을 만족하는 아티스트는 **2명**이고 `rows[0]` 은 **제이드** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3번과 골격은 같고 함수만 avg. 아티스트별 cnt 평균을 WITH 로 낸 뒤 WHERE 로 35 이상만 남긴다.

세부구현:
1. 아티스트→곡(PERFORMS)과 그 곡의 재생(PLAYED)을 한 패턴으로 MATCH 하고 재생 관계에 변수를 붙인다.
2. WITH 로 아티스트와 avg 집계(별칭 평균재생)를 넘기고, 이어지는 WHERE 로 평균재생 35 이상만 남긴다.
3. RETURN 에 아티스트 이름·평균재생을 두고 평균재생 내림차순(같으면 이름순)으로 ORDER BY 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 2, '2행이어야 합니다. 경계값(정확히 35)이 포함되는 부등호를 썼는지 확인하세요'
assert rows[0]['아티스트'] == '제이드', '1위가 다릅니다. 합이 아니라 평균(avg)으로 냈는지 확인하세요'
assert all(r['평균재생'] >= 35 for r in rows), '평균 35 미만인 아티스트가 남아 있습니다'
print('✅ 통과!')

## 8. 아티스트의 최고 인기곡 재생수와 히트곡 표시 (max + SET)
**배경**: 이번엔 **최댓값**(max)과 **파생 속성 저장**(SET)을 한 문제에서 씁니다. 먼저 아티스트마다 가장 많이 재생된 곡의 재생수를 구하고(2번은 합산이었지만 이번엔 최댓값), 이어서 재생수가 많은 곡에 "히트곡" 표시를 붙입니다.

**요구사항**:
- (1) 곡별 총재생수(`sum(p.cnt)`)를 먼저 낸 뒤, 그 곡을 아티스트에 이어(`PERFORMS`) **아티스트별 최댓값**(`max`)을 구해 **`rows`** 에 담으세요. 별칭은 **`아티스트`**·**`최고재생`**. **최고재생 내림차순**(같으면 아티스트 이름 오름차순)으로 정렬하세요.
- (2) 곡별 총재생수가 **100 이상인지**를 **참/거짓 값으로** 그 곡 노드의 **`is_hit`** 속성에 저장하세요(비교식의 결과를 그대로 저장할 수 있습니다). 조건에 맞지 않는 곡도 `False` 가 저장돼야 합니다. 저장한 뒤 DB 에서 `is_hit` 가 참인 곡 수를 세어 **`hit_count`** 에 담으세요.

**예시**: `rows[0]` 은 아티스트 **제이드**, 최고재생 **115** 입니다. `is_hit` 가 참인 곡은 **2개**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- (1) 곡별 합을 먼저 낸 뒤 아티스트로 다시 묶어 최댓값을 고른다. 2번(합산)과 골격은 같고 집계 함수만 max 로 바뀐다.
- (2) 곡별 합을 낸 뒤, 그 합이 기준을 넘는지 비교한 결과를 SET 으로 저장하고 참인 곡을 센다.

세부구현:
1. (1) 청취자→곡(PLAYED)을 MATCH 하고 첫 WITH 로 곡과 곡별 sum(별칭 곡재생)을 넘긴다.
2. 그 곡을 아티스트에 잇는 MATCH(PERFORMS)를 이어 쓰고, 둘째 WITH 로 아티스트와 max(곡재생)(별칭 최고재생)을 넘겨 최고재생 내림차순(같으면 이름순)으로 ORDER BY 한다.
3. (2) 다시 청취자→곡(PLAYED)을 MATCH 하고 WITH 로 곡과 sum(별칭 합)을 넘긴 뒤, SET 으로 곡의 is_hit 에 비교 결과(합이 100 이상인가)를 저장한다.
4. Song 중 is_hit 가 참인 것만 세어(count) hit_count 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows[0]['아티스트'] == '제이드' and rows[0]['최고재생'] == 115, '1위가 다릅니다. 곡별 합을 먼저 낸 뒤 아티스트별 max 를 골랐는지 확인하세요'
assert len(rows) == 3, '아티스트 3명이 모두 나와야 합니다'
assert rows == sorted(rows, key=lambda r: (-r['최고재생'], r['아티스트'])), '정렬을 확인하세요: 최고재생 내림차순, 같으면 아티스트 이름 오름차순'
assert hit_count == 2, '히트 곡 수가 다릅니다. is_hit 를 100 이상 기준으로 저장했는지 확인하세요'
# 파이썬 변수뿐 아니라 DB 를 다시 조회해 is_hit 가 실제로 저장됐는지 확인한다
assert run_cypher("MATCH (s:Song) WHERE s.is_hit RETURN count(s) AS n")[0]['n'] == 2, \
    'is_hit 가 참인 곡 수가 다릅니다. 조회만 하지 말고 SET 까지 실행했는지 확인하세요'
assert run_cypher("MATCH (s:Song {title:'모래성'}) RETURN s.is_hit AS v")[0]['v'] is False, \
    '조건에 맞지 않는 곡에도 False 가 저장돼야 합니다. WHERE 로 거르지 말고 비교식 결과를 그대로 SET 하세요'
assert sorted(r['t'] for r in run_cypher("MATCH (s:Song) WHERE s.is_hit "
                                        "RETURN s.title AS t")) == ['등대', '은하수']
print('✅ 통과!')

## 9. 재생수 등급별 곡 목록 (CASE + collect)
**배경**: 총 재생수를 그대로 늘어놓는 대신 **등급 라벨**로 묶으면 차트가 한눈에 읽힙니다. 숫자를 구간 라벨로 바꾸고, 그 라벨을 그룹핑 키로 삼아 개수와 목록을 한 번에 냅니다.

**요구사항**:
- `(:Listener)-[p:PLAYED]->(s:Song)` 를 곡별 `sum(p.cnt)` 로 합친 뒤, 그 합이 **100 이상이면 `히트`**, **50 이상이면 `인기`**, 그 외는 **`일반`** 이라는 등급을 붙이세요.
- 등급별로 **곡 수**와 **곡 제목 목록**을 구해 **`rows`** 에 담으세요. 별칭은 **`등급`**·**`곡수`**·**`곡목록`**.
- **등급 이름 오름차순**으로 정렬하세요.

**예시**: 결과는 **3행**이고, `히트` 의 `곡수` 는 **2** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 곡별 합을 WITH 로 낸 뒤, 그 합을 조건별 라벨로 바꾼다. 그 라벨은 집계가 아니므로 그대로 그룹핑 키가 되어 개수와 목록이 등급별로 묶인다.

세부구현:
1. MATCH 로 청취자→곡(PLAYED)을 잡고 재생 관계에 변수를 붙인다.
2. WITH 로 곡과 곡별 합을 넘긴다.
3. RETURN 에 조건별 라벨(CASE ... END)을 별칭 등급 으로 두고, 곡 수(count)와 제목 목록(collect)을 별칭 곡수·곡목록 으로 함께 둔다. 조건은 좁은 것(100 이상)부터 위에 쓰고 나머지는 ELSE 로 받는다.
4. ORDER BY 등급 으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
grade = {r['등급']: r for r in rows}
assert set(grade) == {'히트', '인기', '일반'}, '등급 세 가지가 모두 나와야 합니다. ELSE 로 남는 값을 받았는지 확인하세요'
assert grade['히트']['곡수'] == 2 and sorted(grade['히트']['곡목록']) == ['등대', '은하수'], \
    '히트 등급의 곡 수나 목록이 다릅니다. 100 이상 기준과 collect 로 모은 제목을 확인하세요'
assert grade['인기']['곡수'] == 1 and grade['일반']['곡수'] == 3, \
    '인기·일반 등급의 곡 수가 다릅니다. 50 이상 100 미만이 인기, 나머지가 일반입니다'
assert rows == sorted(rows, key=lambda r: r['등급']), '등급 이름 오름차순으로 정렬하세요'
# DB 로 한 번 더 검산: 총재생 100 이상인 곡 수가 '히트' 곡수와 같아야 한다
hit_check = run_cypher("MATCH (:Listener)-[p:PLAYED]->(s:Song) "
                      "WITH s, sum(p.cnt) AS 합 WHERE 합 >= 100 "
                      "RETURN count(s) AS n")[0]['n']
assert grade['히트']['곡수'] == hit_check, \
    '히트 곡수가 DB 를 다시 센 값과 다릅니다. 예시 값을 옮겨 적지 말고 직접 조회하세요'
print('✅ 통과!')

## 10. 접기와 펴기가 서로를 되돌리는지 확인 (collect + UNWIND)
**배경**: 집계로 접어 둔 목록을 다시 한 줄씩 다뤄야 할 때가 많습니다. 접기(`collect`)와 펴기(`UNWIND`)가 정말 서로를 되돌리는지, 두 결과를 나란히 내서 확인합니다.

**요구사항**:
- (1) `(a:Artist)-[:PERFORMS]->(s:Song)` 를 아티스트별로 묶어 곡 제목을 `collect` 로 모으고, 그 리스트의 길이를 `size` 로 세어 **`folded`** 에 담으세요. 별칭은 **`아티스트`**·**`곡목록`**·**`곡수`**, 정렬은 아티스트 이름 오름차순.
- (2) 같은 패턴을 `WITH` 로 모은 뒤 `UNWIND` 로 펼쳐 **`unfolded`** 에 담으세요. 별칭은 **`아티스트`**·**`곡`**, 정렬은 아티스트 이름 오름차순, 같으면 곡 제목 오름차순.

**예시**: `folded` 는 **3행**, `unfolded` 는 **6행**(전체 곡 수)입니다. `folded` 의 `곡수` 를 다 더하면 `unfolded` 의 행 수와 같아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 패턴을 두 번 쓴다. 한 번은 모은 채로, 한 번은 모았다가 다시 펼쳐서.
- 집계 결과를 다음 단계의 입력으로 쓰려면 RETURN 이 아니라 WITH 로 넘겨야 한다.

세부구현:
1. (1) MATCH 로 아티스트→곡(PERFORMS)을 잡고, RETURN 에 아티스트 이름·곡 제목을 모은 리스트·그 리스트의 길이를 별칭과 함께 둔다. 아티스트 이름순으로 정렬한다.
2. (2) 같은 MATCH 뒤에 WITH 로 아티스트 이름과 모은 리스트를 넘긴다.
3. UNWIND 로 그 리스트를 한 줄씩 펼쳐 별칭을 곡 으로 둔다.
4. RETURN 에 아티스트·곡을 두고 아티스트 오름차순, 같으면 곡 오름차순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(folded) == 3, '아티스트마다 한 행씩, 3행이어야 합니다'
assert len(unfolded) == 6, '펼친 결과는 전체 곡 수(6)와 같아야 합니다. UNWIND 대상이 모은 리스트가 맞는지 확인하세요'
assert sum(r['곡수'] for r in folded) == len(unfolded), '접은 곡수의 합과 편 행 수가 다릅니다. 두 쿼리가 같은 패턴을 쓰는지 확인하세요'
assert {r['아티스트']: r['곡수'] for r in folded} == {'루나': 2, '제이드': 3, '카이': 1}, '아티스트별 곡수가 다릅니다. size 로 리스트 길이를 셌는지 확인하세요'
assert [(r['아티스트'], r['곡']) for r in unfolded] == [('루나', '밤하늘'), ('루나', '은하수'), ('제이드', '등대'), ('제이드', '모래성'), ('제이드', '파도'), ('카이', '질주')], '정렬을 확인하세요: 아티스트 오름차순, 같으면 곡 제목 오름차순'
assert folded == sorted(folded, key=lambda r: r['아티스트']), 'folded 도 아티스트 이름 오름차순으로 정렬하세요'
print('✅ 통과!')

## 11. 인덱스가 레이블마다 따로 걸리는지 확인 (EXPLAIN)
**배경**: 교안에서 유전자 이름 조회의 dbHits 가 인덱스 하나로 26,228 에서 3 으로 줄어드는 것을 봤습니다. 그때 함께 본 것이 **레이블을 빼면 그 인덱스를 못 쓴다**는 사실입니다. 여기 노드 열몇 개짜리 그래프에서도 **실행계획의 연산자 이름**으로 같은 일이 확인됩니다. 데이터가 작아도 계획은 정직합니다.

아래 준비 셀의 `explain_plan(쿼리, profile=False)` 은 쿼리를 **실행하지 않고 계획만** 보여 주고, `(연산자 이름 리스트, 총 dbHits)` 를 돌려줍니다.

**요구사항**:
- (1) `Listener` 의 `name` 에 인덱스를 만드세요. 이름은 **`listener_name`**, `IF NOT EXISTS` 를 붙여 여러 번 실행해도 되게 하고, 바로 뒤에 `CALL db.awaitIndexes()` 로 완성될 때까지 기다립니다.
- (2) `explain_plan` 으로 **레이블을 적은** 조회의 계획을 받아 **`ops_labeled`** 에 담으세요. 쿼리는 `MATCH (l:Listener {name: '하늘'}) RETURN l.name AS 이름` 이고 `profile=False` 로 부릅니다.
- (3) 같은 조회에서 **레이블만 뺀** 계획을 받아 **`ops_plain`** 에 담으세요. 쿼리는 `MATCH (x {name: '하늘'}) RETURN x.name AS 이름` 입니다.

> `explain_plan` 은 값 두 개를 돌려줍니다. 계획 리스트만 쓸 것이므로 `ops_labeled, _ = ...` 처럼 받으면 됩니다.

**예시**: 레이블을 적은 쪽에는 `NodeIndexSeek`(색인으로 바로 짚음)가, 뺀 쪽에는 `AllNodesScan`(그래프 전체를 훑음)이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 인덱스를 먼저 만들고, 같은 조회를 레이블 있는 판과 없는 판으로 두 번 재어 계획을 견준다.
- 비교의 근거는 실행 시간이 아니라 계획에 나온 연산자 이름이다.

세부구현:
1. run_cypher 로 인덱스를 만든다(레이블과 속성을 지정하고, 이미 있어도 에러가 안 나게 한다).
2. 이어서 인덱스가 다 만들어질 때까지 기다리는 프로시저를 호출한다.
3. explain_plan 을 profile=False 로 불러 레이블을 적은 조회의 계획을 받는다.
4. 같은 방식으로 레이블을 뺀 조회의 계획을 받는다.
5. 두 리스트를 각각 출력해 어떤 연산자가 쓰였는지 눈으로 견준다.
```

</details>

In [ ]:
# [제공 코드] 실행계획 보기 헬퍼: 실행만 하세요.
# explain_plan(쿼리, profile=False) 는 계획만, True 는 비용(dbHits)까지. quiet=True 면 찍지 않고 값만.
PLAN_CALLS = []   # 부른 기록: (쿼리, profile 여부, 연산자 목록).
#                 과제 채점이 "정말 재 봤는지"와 "그 결과를 담았는지"를 볼 때 쓴다


def explain_plan(query, profile=True, quiet=False, **params):
    """실행계획을 출력하고 (연산자 이름 리스트, 총 dbHits) 를 돌려준다."""
    # profile=True 는 PROFILE(실제로 실행하며 dbHits 측정), False 는 EXPLAIN(실행 없이 계획만)
    with driver.session() as session:
        if profile:
            result = session.run("PROFILE " + query, **params)
            list(result)                      # PROFILE 은 결과를 끝까지 읽어야 계측이 끝난다
            plan = result.consume().profile   # 계획은 실행이 끝난 뒤에야 받을 수 있다
        else:
            plan = session.run("EXPLAIN " + query, **params).consume().plan
    total = 0        # 계획 전체의 dbHits 합
    operators = []   # 위에서부터 만난 연산자 이름들. 뒤에서 'NodeIndexSeek 이 있나' 를 볼 때 쓴다

    # 실행계획은 나무 모양이라, 자기 자신을 다시 부르며 아래로 내려간다
    def walk(node, depth=0):
        nonlocal total
        hits = node.get("dbHits")                   # EXPLAIN 으로 뽑은 계획에는 이 값이 없다(None)
        total += hits or 0
        name = node["operatorType"].split("@")[0]   # 'NodeIndexSeek@neo4j' 에서 이름만 남긴다
        operators.append(name)
        cost = f"| dbHits = {hits}" if profile else ""
        if not quiet:                     # quiet=True 면 재기만 하고 찍지 않는다
            print("  " * depth, name, cost)
        for child in node.get("children", []):
            walk(child, depth + 1)        # 자식 계획은 한 칸 더 들여써 찍는다

    walk(plan)
    # 무엇을 쟀고 무엇이 나왔는지 남긴다(채점이 손으로 적은 목록을 걸러 낼 때 본다)
    PLAN_CALLS.append((query, profile, operators))
    if profile and not quiet:
        print("총 dbHits:", total)
    return operators, total

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 연산자 이름을 손으로 적어 넣어도 값은 맞는다. 그래서 explain_plan 을 정말 불렀는지,
# 그리고 그 결과를 그대로 담았는지 먼저 본다(PLAN_CALLS 에 쿼리와 연산자 목록이 쌓인다)
asked = [q for q, _, _ in PLAN_CALLS]
measured = [ops for _, _, ops in PLAN_CALLS]
assert any(':Listener' in q for q in asked), \
    'explain_plan 으로 레이블을 적은 조회의 계획을 직접 받아 ops_labeled 에 담으세요'
assert any('(x {name' in q for q in asked), \
    'explain_plan 으로 레이블을 뺀 조회의 계획도 직접 받아 ops_plain 에 담으세요'
assert ops_labeled in measured and ops_plain in measured, \
    '두 변수에는 explain_plan 이 돌려준 목록을 그대로 담으세요(손으로 적은 목록은 통과하지 못합니다)'
index_names = [r['name'] for r in run_cypher("SHOW INDEXES YIELD name RETURN name")]
assert 'listener_name' in index_names, 'listener_name 인덱스가 없습니다. 이름을 그대로 listener_name 으로 만들었는지 확인하세요'
assert 'NodeIndexSeek' in ops_labeled, '레이블을 적은 조회가 인덱스를 쓰지 않았습니다. 인덱스를 만든 뒤 db.awaitIndexes() 로 기다렸는지 확인하세요'
assert 'AllNodesScan' in ops_plain, '레이블을 뺀 조회의 계획에 AllNodesScan 이 없습니다. 쿼리에서 레이블만 뺐는지 확인하세요'
assert 'NodeIndexSeek' not in ops_plain, '레이블을 뺐는데도 인덱스를 썼습니다. 쿼리를 다시 확인하세요'
print('✅ 통과!')

## 12. 제목 일부로 찾을 때는 TEXT 인덱스
**배경**: 11번에서 **레이블을 빼면** 인덱스를 못 쓴다는 것을 봤습니다. 인덱스를 못 쓰게 되는 이유가 하나 더 있습니다. **조건의 종류가 그 인덱스로 풀 수 없는 것일 때**입니다.

5번에서 `Song.title` 에 UNIQUE 제약을 걸었고, 제약은 인덱스(종류는 `RANGE`)를 겸합니다. 그런데 제목에 **`하` 가 들어간 곡**을 찾는 조회는 그 인덱스로 짚지 못합니다. 이름을 사전 순으로 늘어놓은 찾아보기라, **앞이 같은 것**은 한자리에 모여 있어도 **가운데에 들어 있는 것**은 사전 어디에나 흩어져 있기 때문입니다.

**요구사항**: 아래 조회 하나를 놓고 인덱스 상태만 바꿔 가며 **실행계획의 연산자**를 두 번 받습니다. 조회는 두 번 모두 **똑같이** 씁니다.

```
MATCH (s:Song) WHERE s.title CONTAINS '하' RETURN count(s) AS 곡수
```

- (1) 지금 상태(제약이 만든 `RANGE` 인덱스만 있음)에서 `explain_plan(쿼리, profile=False)` 로 연산자 목록을 받아 **`ops_range`** 에 담으세요.
- (2) `Song.title` 에 **TEXT 인덱스** **`song_title_text`** 를 만드세요(`CREATE TEXT INDEX 이름 IF NOT EXISTS FOR (s:Song) ON (s.title)`). 만든 뒤 `CALL db.awaitIndexes()` 로 기다립니다.
- (3) **같은 조회**의 연산자 목록을 다시 받아 **`ops_text`** 에 담으세요.
- (4) `SHOW INDEXES` 로 `song_title_text` 의 **`type`** 을 조회해 **`text_type`** 에 문자열로 담으세요(`YIELD name, type` 으로 받아 그 행의 `type` 값만 꺼냅니다).

> `explain_plan` 은 `(연산자 이름 리스트, 총 dbHits)` 두 개를 돌려줍니다. 계획만 쓸 것이므로 `ops_range, _ = ...` 처럼 받으면 됩니다.

**예시**: `ops_range` 에는 `NodeIndexScan`(인덱스에 실린 제목을 처음부터 끝까지 훑음)이, `ops_text` 에는 `NodeIndexContainsScan`(부분일치 전용으로 바로 짚음)이 들어 있습니다. `text_type` 은 `'TEXT'` 입니다. 참고로 이 조회에 걸리는 곡은 **['밤하늘', '은하수']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 11번과 같은 모양이다. 쿼리는 그대로 두고 인덱스 상태만 바꿔 계획을 두 번 받는다.
- 종류가 다른 인덱스는 같은 속성에 함께 둘 수 있다. RANGE 를 지울 필요가 없다.

세부구현:
1. 조회 문자열을 변수 하나에 담아 두 번 다 그것을 쓴다(글자가 달라지면 대조가 성립하지 않는다).
2. explain_plan(쿼리, profile=False) 의 첫 번째 반환값을 ops_range 에 담는다.
3. CREATE TEXT INDEX song_title_text IF NOT EXISTS FOR (s:Song) ON (s.title) 를 실행하고
   CALL db.awaitIndexes() 로 기다린다.
4. 같은 쿼리로 2번을 한 번 더 해서 ops_text 에 담는다.
5. SHOW INDEXES YIELD name, type 에서 song_title_text 행의 type 값을 text_type 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 계획을 재지 않고 연산자 이름만 적어도 값은 맞는다. 같은 쿼리를 두 번 쟀는지,
# 그리고 두 변수가 그때 받은 목록 그대로인지 본다(손으로 적으면 여기서 걸린다)
asked = [q for q, _, _ in PLAN_CALLS]
measured = [ops for _, _, ops in PLAN_CALLS]
assert sum(1 for q in asked if 'CONTAINS' in q.upper()) >= 2, \
    '같은 부분일치 조회를 TEXT 인덱스 만들기 전과 후로 두 번 재어 ops_range·ops_text 에 담으세요'
assert ops_range in measured, \
    'ops_range 가 explain_plan 이 돌려준 목록이 아닙니다. TEXT 인덱스를 만들기 전에 재어 그 결과를 그대로 담으세요'
assert ops_text in measured, \
    'ops_text 도 explain_plan 이 돌려준 목록을 그대로 담으세요'
assert 'NodeIndexScan' in ops_range, 'RANGE 인덱스만 있을 때는 계획에 NodeIndexScan 이 나와야 합니다. 5번의 UNIQUE 제약을 먼저 만들었는지, 쿼리를 그대로 썼는지 확인하세요'
assert 'NodeIndexContainsScan' in ops_text, 'TEXT 인덱스를 만든 뒤 계획에 NodeIndexContainsScan 이 없습니다. CREATE TEXT INDEX 로 만들고 db.awaitIndexes() 로 기다렸는지 확인하세요'
assert 'NodeIndexContainsScan' not in ops_range, '첫 번째 계획이 이미 부분일치 전용이었습니다. TEXT 인덱스를 만들기 전에 먼저 쟀는지 확인하세요'
assert text_type == 'TEXT', "인덱스 종류가 TEXT 가 아닙니다. CREATE INDEX 가 아니라 CREATE TEXT INDEX 로 만들었는지 확인하세요"
# DB 를 다시 조회해 두 인덱스가 같은 속성에 나란히 살아 있는지 확인한다
live = run_cypher("SHOW INDEXES YIELD name, type, properties, state "
                  "WHERE type <> 'LOOKUP' RETURN name, type, properties, state")
on_title = [r for r in live if r['properties'] == ['title']]
assert sorted(r['type'] for r in on_title) == ['RANGE', 'TEXT'], '같은 title 속성에 RANGE 와 TEXT 가 함께 있어야 합니다(제약이 만든 RANGE 를 지우지 마세요)'
assert all(r['state'] == 'ONLINE' for r in on_title), '인덱스 상태가 ONLINE 이 아닙니다. CALL db.awaitIndexes() 를 불렀는지 확인하세요'
print('✅ 통과!')

## 13. 아티스트마다 인기곡 2곡씩 (그룹별 상위 N)
**배경**: 1번은 **전체에서** 상위 3곡을 뽑았습니다. 차트 화면에서 더 자주 쓰는 것은 "**아티스트마다** 대표곡 두 곡씩" 입니다. `LIMIT` 은 전체에서 자르므로 이 물음에는 맞지 않습니다. **줄을 세운 뒤 `collect` 로 접어 앞부분만 자르면** 그룹마다 잘립니다.

**요구사항**:
- (1) `(a:Artist)-[:PERFORMS]->(s:Song)<-[p:PLAYED]-(:Listener)` 에서 **곡별 총재생수**(`sum(p.cnt)`)를 먼저 낸 뒤, 그 값으로 **곡재생 내림차순**(같으면 곡 제목 오름차순)으로 줄을 세우세요.
- 그 순서 그대로 아티스트별로 접어, **곡 수**와 **인기곡 상위 2곡의 제목 리스트**를 내어 **`rows`** 에 담으세요. 별칭은 **`아티스트`**·**`곡수`**·**`인기곡`**. 리스트를 자를 때는 `collect(s.title)[0..2]` 를 씁니다.
- **아티스트 이름 오름차순**으로 정렬하세요.
- (2) 같은 "아티스트별 곡 제목 목록"을 이번에는 **패턴 컴프리헨션**으로 한 줄에 받아 **`pattern_rows`** 에 담으세요. `MATCH (a:Artist)` 만 잡고 `RETURN` 자리에서 `[(a)-[:PERFORMS]->(t:Song) | t.title]` 로 목록을 만듭니다. 별칭은 **`아티스트`**·**`곡목록`**, 정렬은 **아티스트 이름 오름차순**.
- (3) (2)의 목록에서 **제목이 두 글자인 곡만** 남긴 리스트를 함께 내어 **`short_rows`** 에 담으세요. 같은 `MATCH (a:Artist)` 에 **리스트 컴프리헨션**을 씁니다(`[x IN 목록 WHERE 조건]`). 조건은 `size(x) = 2` 이고, 별칭은 **`아티스트`**·**`두글자곡`**, 정렬은 **아티스트 이름 오름차순**.

**예시**: (1) `rows` 는 **3행**이고, 아티스트 **제이드** 의 `인기곡` 은 **['등대', '파도']** 입니다(곡수 3 중 상위 2곡). (2) `pattern_rows` 도 3행이고, 각 행의 `곡목록` 길이는 (1)의 `곡수` 와 같습니다(패턴 컴프리헨션은 담기는 순서를 보장하지 않으므로 **길이로** 견줍니다). (3) `short_rows` 의 `두글자곡` 을 모두 모으면 **['등대', '질주', '파도']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- ORDER BY 는 줄을 세울 뿐 자르지 않는다. 세운 순서 그대로 collect 로 접으면 리스트 앞쪽이 곧 상위다.
- ORDER BY 가 collect 보다 위에 있어야 한다. 아래에 두면 이미 접힌 뒤라 리스트 안 순서가 안 바뀐다.

세부구현:
1. WITH 로 아티스트와 곡을 넘기면서 곡별 재생 합을 먼저 낸다.
2. 그 합으로 ORDER BY 를 걸어 줄을 세운다(같으면 곡 제목순).
3. 다시 WITH 로 아티스트별로 접는다. 곡 수는 count, 인기곡은 collect 한 리스트를 요구사항의 자르기 표기로 앞 두 개만 남긴다.
4. RETURN 에 요구사항의 별칭을 두고 아티스트 이름순으로 정렬한다.
5. (2)는 MATCH (a:Artist) 만 잡고, RETURN 자리에 대괄호 패턴을 적어 목록을 만든다.
6. (3)은 그 목록을 한 번 더 대괄호로 감싸 조건을 건다: [x IN 목록 WHERE size(x) = 2].
   세로줄 뒤 식을 안 적으면 걸러 낸 값이 그대로 담긴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 3, '아티스트마다 한 행씩 나와야 합니다'
assert all(len(r['인기곡']) <= 2 for r in rows), '인기곡 리스트는 최대 2개여야 합니다. collect(...)[0..2] 로 잘랐는지 확인하세요'
expected_topn = {'루나': ['은하수', '밤하늘'], '제이드': ['등대', '파도'], '카이': ['질주']}
assert {r['아티스트']: r['인기곡'] for r in rows} == expected_topn, '인기곡이 다릅니다. 곡별 합을 먼저 낸 뒤 곡재생 내림차순(같으면 제목순)으로 정렬하고 그 다음에 collect 로 접었는지 확인하세요'
expected_cnt = {'루나': 2, '제이드': 3, '카이': 1}
assert {r['아티스트']: r['곡수'] for r in rows} == expected_cnt, '곡수가 다릅니다. 재생 행 수가 아니라 곡 수를 셌는지 확인하세요'
assert rows == sorted(rows, key=lambda r: r['아티스트']), '아티스트 이름 오름차순으로 정렬하세요'
assert len(pattern_rows) == 3, 'pattern_rows 도 아티스트마다 한 행이어야 합니다'
assert {r['아티스트']: len(r['곡목록']) for r in pattern_rows} == expected_cnt, '패턴 컴프리헨션이 만든 곡목록의 길이가 (1)의 곡수와 달라야 할 이유가 없습니다. 패턴을 [(a)-[:PERFORMS]->(t:Song) | t.title] 로 적었는지 확인하세요'
assert len(short_rows) == 3, 'short_rows 도 아티스트마다 한 행이어야 합니다'
assert sorted(t for r in short_rows for t in r['두글자곡']) == ['등대', '질주', '파도'], '두글자곡이 다릅니다. [x IN 목록 WHERE size(x) = 2] 로 걸렀는지 확인하세요'
assert all(all(len(t) == 2 for t in r['두글자곡']) for r in short_rows), '두 글자가 아닌 제목이 남아 있습니다'
print('✅ 통과!')

## 14. 함께 들은 곡 추천: 원점수와 비율
**배경**: "`등대` 을 들은 사람이 **함께 들은 다른 곡**" 을 추천해 봅니다. 가운데에 청취자를 두고 세는 공유 패턴입니다. 그런데 **무엇으로 줄 세울지**에 따라 답이 달라집니다. 두 점수를 만들어 견줍니다.

**요구사항**:
- (1) **원점수**: `(s:Song {title: '등대'})<-[:PLAYED]-(l:Listener)-[:PLAYED]->(other:Song)` 에서 자기 자신(`other <> s`)을 뺀 뒤, 곡마다 **서로 다른 청취자 수**(`count(DISTINCT l)`)를 세어 **`raw`** 에 담으세요. 별칭은 **`곡`**·**`공유청취자수`**. **공유청취자수 내림차순**(같으면 곡 제목 오름차순)으로 정렬합니다.
- (2) **비율**: 같은 패턴에 후보 곡의 **전체 청취자 수**를 이어 붙여 `공유청취자수 / 전체청취자수` 를 내어 **`ratio`** 에 담으세요. 별칭은 **`곡`**·**`공유청취자수`**·**`전체청취자수`**·**`겹침비율`** 이고, `겹침비율` 은 **소수 셋째 자리까지 `round`** 합니다. **겹침비율 내림차순**(같으면 공유청취자수 내림차순, 그다음 곡 제목 오름차순)으로 정렬합니다.
  - 전체 청취자 수는 1차 집계 뒤에 `MATCH (other)<-[:PLAYED]-(x:Listener)` 를 이어 `count(DISTINCT x)` 로 셉니다.
  - ⚠️ **정수끼리 나누면 소수점 아래가 버려져 거의 전부 `0` 이 됩니다.** 한쪽을 `toFloat` 로 감싸세요(`round(toFloat(공유청취자수) / 전체청취자수, 3)`).

**예시**: (1) `raw[0]` 은 곡 **은하수**(공유청취자수 2) 입니다. (2) `ratio[0]` 은 곡 **모래성**(공유 1 / 전체 1 = **1.0**) 입니다. **두 1위가 서로 다릅니다.**

<details><summary>힌트</summary>

```text
접근방법:
- 교안_02 의 공유 패턴 랭킹을 곡으로 옮긴다(대상곡 ← 청취자 → 다른곡).
- 비율은 1차 집계로 공유 수를 낸 뒤, 그 결과에 후보의 전체 수를 다시 이어 붙여 나눈다.

세부구현:
1. (1) MATCH 로 공유 패턴을 잡고 WHERE 로 자기 자신을 뺀 뒤 count(DISTINCT l) 로 센다.
2. (2) 같은 패턴에서 WITH other, count(DISTINCT l) AS 공유청취자수 로 1차 집계한다.
3. 이어서 MATCH (other)<-[:PLAYED]-(x:Listener) 로 후보의 전체 청취자를 잡고
   WITH other, 공유청취자수, count(DISTINCT x) AS 전체청취자수 로 2차 집계한다.
4. RETURN 에서 round(toFloat(공유청취자수) / 전체청취자수, 3) 을 겹침비율로 낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert [(r['곡'], r['공유청취자수']) for r in raw] == [('은하수', 2), ('모래성', 1)], '원점수 랭킹이 다릅니다. 자기 자신을 뺐는지, 청취자를 DISTINCT 로 셌는지 확인하세요'
assert [(r['곡'], r['공유청취자수'], r['전체청취자수'], r['겹침비율']) for r in ratio] == [('모래성', 1, 1, 1.0), ('은하수', 2, 3, 0.667)], '비율 랭킹이 다릅니다. 전체 청취자 수를 1차 집계 뒤에 이어 붙였는지, toFloat 로 감싸 나눴는지, 정렬 기준을 그대로 따랐는지 확인하세요'
# 정수 나눗셈을 그대로 쓴 답안은 여기서 걸린다(비율이 전부 0 이나 1 이 된다)
assert any(0 < r['겹침비율'] < 1 for r in ratio), '겹침비율이 전부 0 또는 1 입니다. 정수끼리 나눠 소수점이 버려졌습니다. toFloat 를 쓰세요'
assert raw[0]['곡'] != ratio[0]['곡'], '두 랭킹의 1위가 같습니다. 원점수와 비율을 서로 다른 기준으로 정렬했는지 확인하세요'
print('✅ 통과!')

## 15. 한 칸에 든 두 값 갈라 쓰기 (trim·split·toInteger)
**배경**: 곡 노드에는 준비 셀이 넣어 둔 **`tag`** 속성이 있습니다. 외부 시스템에서 받은 문자열을 손대지 않고 그대로 넣은 것이라 **`" 발라드|2021 "`** 처럼 생겼습니다. 한 칸에 **장르와 발매연도 두 값**이 들어 있고, 앞뒤에 **공백**이 붙은 것도 있습니다. 실무에서 받는 원본은 대개 이렇습니다. 쓰려면 먼저 다듬어야 합니다.

**쓸 함수** (교안_01 7-2 에서 본 것들입니다)

| 함수 | 하는 일 |
|---|---|
| `trim(문자열)` | 앞뒤 공백을 뗀다 |
| `split(문자열, 구분자)` | 구분자로 잘라 **리스트**로 만든다 |
| `목록[0]` · `목록[1]` | 리스트에서 자리 번호로 꺼낸다(0 부터 센다) |
| `toInteger(문자열)` | 숫자로 바꾼다. **문자열 `'2022'` 와 숫자 `2022` 는 비교되지 않는다** |

`tag` 를 가르는 구분자는 세로줄(`|`)입니다. 예를 들어 `split(trim(s.tag), '|')` 은 `" 발라드|2021 "` 를 `['발라드', '2021']` 로 만듭니다.

**요구사항**:
- **(1) 장르별 집계**: 곡별 총재생수(`sum(p.cnt)`)를 먼저 낸 뒤, `tag` 에서 **장르**만 꺼내 장르별 **곡 수**와 **총재생수**를 구해 **`rows`** 에 담으세요. 별칭은 **`장르`**·**`곡수`**·**`총재생`**. **총재생 내림차순, 같으면 장르 이름 오름차순**으로 정렬합니다.
  - 장르 앞뒤에 공백이 남으면 안 됩니다(`' 발라드'` 가 아니라 `'발라드'`).
- **(2) 최근 발매곡**: `tag` 에서 **발매연도**를 꺼내 **2022년 이상**인 곡의 **제목만 담은 문자열 리스트**를 **제목 오름차순**으로 **`recent`** 에 담으세요.

**예시**: (1) `rows` 는 **3행**이고 1위는 **댄스**(곡수 2 · 총재생 190)입니다. 곡수를 모두 더하면 전체 곡 수와 같습니다. (2) `recent` 는 **['등대', '질주', '파도']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- (1) 곡별 합을 WITH 로 먼저 낸 뒤, 그 곡의 tag 에서 장르를 꺼내 그룹핑 키로 쓴다.
  장르는 집계가 아니므로 그대로 묶는 기준이 된다(1번·9번에서 라벨을 쓴 자리와 같다).
- 다듬는 순서를 생각한다: 공백을 먼저 떼고 자를지, 자른 뒤 조각을 뗄지. 둘 다 된다.
- (2) 연도 조각은 아직 문자열이다. 숫자로 바꾸지 않고 부등호로 비교하면 아무 행도 안 남는다.

세부구현:
1. MATCH 로 청취자→곡(PLAYED) 을 잡고 WITH 로 곡과 곡별 합(sum)을 넘긴다.
2. RETURN 에 tag 를 다듬어 꺼낸 장르를 별칭 장르 로 두고, 곡 수(count)와 합(sum)을 함께 둔다.
3. ORDER BY 로 총재생 내림차순, 같으면 장르 오름차순 정렬한다.
4. (2) 는 MATCH (s:Song) 만 잡고 WITH 로 연도를 만든 뒤 WHERE 로 거르고, 제목만 리스트로 뽑는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert [(r['장르'], r['곡수'], r['총재생']) for r in rows] == [('댄스', 2, 190), ('발라드', 3, 150), ('록', 1, 35)], '장르별 집계가 다릅니다. 공백을 떼었는지(trim), 총재생 내림차순(같으면 장르순)인지 확인하세요'
assert all(g == g.strip() for g in (r['장르'] for r in rows)), '장르에 공백이 남아 있습니다. trim 으로 앞뒤 공백을 떼세요'
assert sum(r['곡수'] for r in rows) == 6, '곡수의 합이 전체 곡 수와 다릅니다. 곡별 합을 WITH 로 먼저 냈는지 확인하세요'
assert recent == ['등대', '질주', '파도'], '최근 발매곡이 다릅니다. 연도를 toInteger 로 바꿔 2022 이상만 남겼는지, 제목 오름차순인지 확인하세요'
print('✅ 통과!')

---
수고했어요! LV2 에서 랭킹·합산·HAVING·collect·UNWIND·UNIQUE 제약·2단 파이프라인·최댓값(max)·파생 속성(SET)·조건 라벨(CASE)·인덱스(레이블별·종류별)·그룹별 상위 N·비율 점수·문자열 다듬기(`trim`·`split`·`toInteger`)를 **조합**했습니다. LV3 에서는 이 모두를 엮어 **추천 엔진과 판매 리포트**를 만듭니다.